# 003 SPEC Modeling Credit Card Fraud

## Header

In [214]:
from notebook_config import setup

In [215]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import settings

from src.processing.upload_sot import SOT_BUCKET_KEY
from src.utils.s3_client import get_s3_client

In [216]:
import logging
logger = logging.getLogger(__name__)

## Dataset: Credit Card Fraud SOT

In [217]:
bucket = settings.MINIO_BUCKET

s3 = get_s3_client()
obj = s3.get_object(Bucket=bucket, Key=SOT_BUCKET_KEY)
payload = obj['Body'].read()

df = pd.read_parquet(io.BytesIO(payload))


2026-05-03 19:04:04 | INFO | src.utils.s3_client | Creating S3 client with endpoint: http://localhost:9000
2026-05-03 19:04:04 | INFO | src.utils.s3_client | S3 client created successfully


In [218]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,transaction_id
0,0,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0,16b96551-2c3b-5141-96e6-941df3c5fe6f
1,0,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0,dda1d3f0-a6ae-5f23-b1e9-abcedbbbbb0f
2,1,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0,149991fc-71eb-530f-8264-9943d7d3a57e
3,1,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0,a297d023-34a8-5b28-ad2f-bca60d04b752
4,2,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0,4d1ec995-cd07-5337-83e0-1a5a56bf2783


In [219]:
df.shape

(284807, 32)

In [220]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class', 'transaction_id'],
      dtype='object')

## Modelando SPEC a partir de SOT

In [221]:
print(df['Time'].dtype)

int64


In [222]:
df['Time'].value_counts()

Time
163152    36
64947     26
68780     25
3767      21
3770      20
          ..
172760     1
172758     1
172757     1
172756     1
172754     1
Name: count, Length: 124592, dtype: int64

In [223]:
df['Time'].describe()

count   284,807.0000
mean     94,813.8596
std      47,488.1460
min           0.0000
25%      54,201.5000
50%      84,692.0000
75%     139,320.5000
max     172,792.0000
Name: Time, dtype: float64

### Features Temporais

In [224]:
df['t_sec'] = df['Time']

In [225]:
df['t_hour'] = df['t_sec'] / 3600

In [226]:
df['t_day'] = df['t_hour'] / 24

In [227]:
df['hour_of_day'] = ((df['t_sec'] // 3600) % 24).astype(int)

In [228]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)

In [229]:
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)

In [230]:
df['t_dt'] = pd.to_datetime(df['t_sec'], unit='s')

In [231]:
df['tx_count_last_1h'] = (
    df.set_index('t_dt')['transaction_id'].rolling('1h').count().shift(1).values
)

### Features Baseadas no valor (`Amount`)

In [232]:
Z_SCORE_GUARDRAIL = 1e-9
LOW_QUANTILE = 0.10
HIGH_QUANTILE = 0.90

In [233]:
df['amount_log'] = np.log1p(df['Amount'])

In [234]:
df['amount_zscore'] = (df['Amount'] - df['Amount'].mean()) / (df['Amount'].std() + Z_SCORE_GUARDRAIL)

In [235]:
df['amount_is_low'] = (df['Amount'] < df['Amount'].quantile(LOW_QUANTILE)).astype(int)

In [236]:
df['amount_is_high'] = (df['Amount'] > df['Amount'].quantile(HIGH_QUANTILE)).astype(int)

In [237]:
df['avg_amount_last_1h'] = (
    df.set_index('t_dt')['amount_log'].rolling('1h').mean().shift(1).values
)

### Magnetude dos Valores `V1` ~ `V28`

In [238]:
v_cols = [f'V{i}' for i in range(1, 29)]

In [239]:
df['v_l2_norm'] = np.sqrt(np.sum(df[v_cols] ** 2, axis=1))

In [240]:
df['v_l1_norm'] = df[v_cols].abs().sum(axis=1)

In [241]:
df['v_max_abs'] = df[v_cols].abs().max(axis=1)

### Extremos por linha

In [242]:
absV = df[v_cols].abs()

In [243]:
thr = absV.stack().quantile(HIGH_QUANTILE)

In [244]:
df['v_extreme_count'] = (absV > thr).sum(axis=1)

### Estaticas por linhas `V1` ~ `V28`

In [245]:
df['v_mean'] = df[v_cols].mean(axis=1)

In [246]:
df['v_std'] = df[v_cols].std(axis=1)

In [247]:
df['v_skew_proxy'] = ((df[v_cols] - df['v_mean'].values.reshape(-1,1))**3).mean(axis=1)

### Valores mais importantes `V1` + `V28`

In [248]:
from sklearn.ensemble import RandomForestClassifier

df = df.sort_values('Time')
cut = int(len(df) * 0.8)
train = df.iloc[:cut]

X_train = train[v_cols + ['amount_log']]
y_train = train['Class']

In [249]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [250]:
imp = (
    pd.Series(rf.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

top_v = [c for c in imp.index if c.startswith('V')][:4]
print(top_v)

['V14', 'V10', 'V12', 'V4']


In [251]:
for c in top_v:
    df[f'{c}_x_amount_log'] = df[c] * df['amount_log']
    df[f'{c}_div_amount'] = df[c] / (df['amount_log'] + 1e-6)

### SPEC Final

In [253]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,transaction_id,t_sec,t_hour,t_day,hour_of_day,hour_sin,hour_cos,t_dt,tx_count_last_1h,amount_log,amount_zscore,amount_is_low,amount_is_high,avg_amount_last_1h,v_l2_norm,v_l1_norm,v_max_abs,v_extreme_count,v_mean,v_std,v_skew_proxy,V14_x_amount_log,V14_div_amount,V10_x_amount_log,V10_div_amount,V12_x_amount_log,V12_div_amount,V4_x_amount_log,V4_div_amount
0,0,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0,16b96551-2c3b-5141-96e6-941df3c5fe6f,0,0.0000,0.0000,0,0.0000,1.0000,1970-01-01 00:00:00,NaN,5.0148,0.2450,0,0,NaN,3.9116,13.1862,2.5363,2,0.1101,0.7444,0.4760,-1.5604,-0.0621,0.4553,0.0181,-3.0981,-0.1232,6.9111,0.2748
1,0,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0,dda1d3f0-a6ae-5f23-b1e9-abcedbbbbb0f,0,0.0000,0.0000,0,0.0000,1.0000,1970-01-01 00:00:00,1.0000,1.3056,-0.3425,0,0,5.0148,2.6745,9.3470,1.6127,1,0.1586,0.4887,0.1487,-0.1877,-0.1101,-0.2180,-0.1279,1.3908,0.8159,0.5851,0.3432
2,1,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0,149991fc-71eb-530f-8264-9943d7d3a57e,1,0.0003,0.0000,0,0.0000,1.0000,1970-01-01 00:00:01,2.0000,5.9393,1.1607,0,1,3.1602,6.0805,23.9448,2.8901,6,0.0390,1.1695,-0.7382,-0.9856,-0.0279,1.2332,0.0350,0.3925,0.0111,2.2556,0.0639
3,1,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0,a297d023-34a8-5b28-ad2f-bca60d04b752,1,0.0003,0.0000,0,0.0000,1.0000,1970-01-01 00:00:01,3.0000,4.8243,0.1405,0,0,4.0866,4.2844,16.5773,1.9658,2,-0.0861,0.8199,0.3906,-1.3890,-0.0597,-0.2651,-0.0114,0.8598,0.0369,-4.1648,-0.1789
4,2,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0,4d1ec995-cd07-5337-83e0-1a5a56bf2783,2,0.0006,0.0000,0,0.0000,1.0000,1970-01-01 00:00:02,4.0000,4.2625,-0.0734,0,0,4.2710,3.5651,15.0948,1.5487,1,0.1921,0.6576,-0.0415,-4.7726,-0.2627,3.2100,0.1767,2.2941,0.1263,1.7179,0.0946


In [252]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class', 'transaction_id', 't_sec', 't_hour', 't_day', 'hour_of_day',
       'hour_sin', 'hour_cos', 't_dt', 'tx_count_last_1h', 'amount_log',
       'amount_zscore', 'amount_is_low', 'amount_is_high',
       'avg_amount_last_1h', 'v_l2_norm', 'v_l1_norm', 'v_max_abs',
       'v_extreme_count', 'v_mean', 'v_std', 'v_skew_proxy',
       'V14_x_amount_log', 'V14_div_amount', 'V10_x_amount_log',
       'V10_div_amount', 'V12_x_amount_log', 'V12_div_amount',
       'V4_x_amount_log', 'V4_div_amount'],
      dtype='object')

## Baseline

In [254]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, classification_report
from lightgbm import LGBMClassifier  # se não tiver, te passo versão com RandomForest

df = df.sort_values('Time').reset_index(drop=True)

Features vs Target

In [255]:
target = 'Class'
drop_cols = ['Class', 't_dt', 'transaction_id']  # ajuste se necessário
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[target].copy()

Split Temporal (70/15/15)

In [256]:
n = len(df)
i1 = int(n * 0.70)
i2 = int(n * 0.85)

X_train, y_train = X.iloc[:i1], y.iloc[:i1]
X_val, y_val     = X.iloc[i1:i2], y.iloc[i1:i2]
X_test, y_test   = X.iloc[i2:], y.iloc[i2:]

Pesos para desbalanceamento

In [257]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / max(pos, 1)

Baseline

In [258]:
model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight=scale_pos_weight
)
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 384, number of negative: 198980
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034102 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13099
[LightGBM] [Info] Number of data points in the train set: 199364, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001926 -> initscore=-6.250317
[LightGBM] [Info] Start training from score -6.250317
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,400
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


Validação (PR-AUC)

In [259]:
val_prob = model.predict_proba(X_val)[:, 1]
pr_auc = average_precision_score(y_val, val_prob)
print(f"VAL PR-AUC: {pr_auc:.4f}")

VAL PR-AUC: 0.0213


Threshold de F1

In [260]:
prec, rec, thr = precision_recall_curve(y_val, val_prob)
f1 = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = np.argmax(f1)
best_thr = thr[best_idx]
print(f"Best threshold (val): {best_thr:.4f} | F1: {f1[best_idx]:.4f}")

Best threshold (val): 1.0000 | F1: 0.0506


Avaliação

In [261]:
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_thr).astype(int)
print("TEST PR-AUC:", average_precision_score(y_test, test_prob))
print(classification_report(y_test, test_pred, digits=4))

TEST PR-AUC: 0.017958957645301917
              precision    recall  f1-score   support

           0     0.9997    0.9640    0.9815     42670
           1     0.0241    0.7308    0.0467        52

    accuracy                         0.9637     42722
   macro avg     0.5119    0.8474    0.5141     42722
weighted avg     0.9985    0.9637    0.9804     42722

